# Cloud-type classification with a Gaussian Mixture Model

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Extract calibrated temperatures

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()

field = session.extract_field('''
data = loadADDEImage(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
                     size='ALL', unit='TEMP', mag=(-4, -4))
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Display', data)
''')
tb = field.masked()
print(field.shape, field.unit, '%.1f..%.1f' % (np.nanmin(tb), np.nanmax(tb)))

## 2. Features: temperature, gradient, texture

In [ ]:
from scipy.ndimage import sobel, uniform_filter

filled = np.where(np.isfinite(tb), tb, 300.0)
grad = np.hypot(sobel(filled, 0), sobel(filled, 1))
mean = uniform_filter(filled, 7)
texture = np.sqrt(np.clip(uniform_filter(filled**2, 7) - mean**2, 0, None))

m = np.isfinite(tb)
X = np.stack([tb[m], grad[m], texture[m]], axis=-1)
X = (X - X.mean(0)) / (X.std(0) + 1e-6)
print('features:', X.shape)

## 3. Fit the GMM

In [ ]:
from sklearn.mixture import GaussianMixture

k = 5
rng = np.random.default_rng(0)
sub = X[rng.choice(len(X), size=min(40000, len(X)), replace=False)]
gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=0).fit(sub)
lab = gmm.predict(X)

order = np.argsort([tb[m][lab == c].mean() for c in range(k)])
lab = np.argsort(order)[lab]
classes = np.full(tb.shape, np.nan); classes[m] = lab

for c in range(k):
    sel = tb[m][lab == c]
    print('class %d: mean %6.1f K  texture %.3f  (n=%d)' % (
        c, sel.mean(), texture[m][lab == c].mean(), sel.size))

## 4. Plot

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(tb, cmap='inferno_r'); ax[0].set_title('Tb (K)'); ax[0].axis('off')
fig.colorbar(im, ax=ax[0], fraction=0.046)
im2 = ax[1].imshow(classes, cmap='turbo'); ax[1].set_title('GMM cloud regimes')
ax[1].axis('off'); fig.colorbar(im2, ax=ax[1], fraction=0.046)
plt.tight_layout()

## 5. Push back

In [ ]:
from scipy.interpolate import griddata

def regrid(field, values, nlat=180, nlon=300, fill=np.nan):
    m = field.valid & np.isfinite(values)
    pts = np.column_stack([field.lats[m], field.lons[m]])
    glats = np.linspace(np.nanmax(field.lats[m]), np.nanmin(field.lats[m]), nlat)
    glons = np.linspace(np.nanmin(field.lons[m]), np.nanmax(field.lons[m]), nlon)
    GLA, GLO = np.meshgrid(glats, glons, indexing='ij')
    g = griddata(pts, values[m], (GLA, GLO), method='linear')
    return np.where(np.isfinite(g), g, fill).astype('f4'), glats, glons

grid, glats, glons = regrid(field, classes, fill=-1)
session.run('''
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setLayerLabel(label='GMM cloud regimes (Tb + texture)')
''', arrays={'g': (grid, glats, glons)})